# Fine-tuning de Qwen2.5-3B-Instruct para OmniCivitas

Notebook de Colab (GPU T4 gratuita) para el fine-tuning QLoRA descripto en la tesis
(`chapter04.tex`, "Entrenamiento del modelo de lenguaje") y en `finetuning/DEVLOG.md`
del repo de código.

**Antes de correr esto**: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución`
→ GPU T4.

Qué hace, en orden:
1. Clona el repo y regenera el dataset (MASSIVE + MULTI3NLU++ + manual) corriendo los
   mismos scripts de `finetuning/` que se usan en local — así el dataset de Colab es
   exactamente reproducible, no un archivo subido a mano.
2. Instala Unsloth y carga Qwen2.5-3B-Instruct en 4 bits.
3. Agrega adaptadores LoRA con los hiperparámetros exactos comprometidos en la tesis.
4. Entrena 3 épocas sobre `train.jsonl`, valida contra `validacion.jsonl`.
5. Exporta a GGUF para servir con Ollama.
6. Mide los tres indicadores de la tesis sobre `prueba_tool_calling.jsonl` (el held-out
   que nunca participó del entrenamiento).

## 1. Clonar el repo y regenerar el dataset

In [ ]:
import os

if not os.path.isdir("/content/OmniCivitas"):
    !git clone https://github.com/Schonp/OmniCivitas-PFI.git /content/OmniCivitas

%cd /content/OmniCivitas/finetuning
!pip install -q -r requirements.txt


`data/` está en `.gitignore` (son archivos generados, no fuente) — por eso se
regeneran acá en vez de subirse a mano. `ejemplos_tool_calling.jsonl` y
`ejemplos_generados.jsonl` sí viajan con el repo, son la parte escrita/generada
a mano que ningún script puede reconstruir.

In [ ]:
!python preparar_massive.py
!python preparar_multi3nlu.py
!python generar_ejemplos.py
!python armar_dataset.py


## 2. Instalar Unsloth

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2


## 3. Cargar Qwen2.5-3B-Instruct en 4 bits

In [ ]:
from unsloth import FastLanguageModel
import torch

# Las instrucciones son de un solo turno (una frase del ciudadano); no hace
# falta una ventana de contexto grande como la que usa Qwen en producción.
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")


## 4. Adaptadores LoRA

Hiperparámetros tal como quedaron comprometidos en `chapter04.tex` ("Entrenamiento del modelo de lenguaje"): cuantización NF4 4 bits (ya aplicada al cargar el modelo arriba), rank 16, alpha 32, dropout 0.05, sobre las proyecciones de atención (`q/k/v/o_proj`) y feed-forward (`gate/up/down_proj`).

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)


## 5. Dataset → conversaciones

El esquema de `finetuning/esquema.py` (`instruccion`, `intencion`, `confianza`,
`llamada_herramienta`, `respuesta`) no es un formato de chat todavía — hay que
convertir cada fila a una conversación de 3 turnos (system/user/assistant) antes
de aplicar el chat template de Qwen.

**Simplificación conocida, documentada en `DEVLOG.md`**: en producción, intake
(clasifica intención) y ejecutor (tool calling + respuesta) son dos agentes y dos
prompts separados (`app/agents/intake.py`, `app/agents/ejecutor.py`). Este dataset
los combina en un solo turno — una sola llamada que clasifica, decide la tool y
redacta la respuesta — porque así está definido el formato de entrenamiento en la
tesis. El prompt de abajo es una versión resumida del `SYSTEM_EJECUTOR` real
(`app/agents/ejecutor.py`), no el texto completo: no modela la identificación del
ciudadano en el primer contacto ni la derivación a operador, que dependen de un
estado de conversación que este dataset de un solo turno no tiene.

In [ ]:
import json

SYSTEM_PROMPT = (
    'Sos el agente de atención ciudadana de un organismo público argentino.\n'
    'Para cada mensaje del ciudadano, devolvé un único objeto JSON con esta forma '
    'exacta, sin texto fuera del JSON:\n\n'
    '{"intencion": "consulta|reclamo|reserva|tramite|otro", "confianza": 0.0 a 1.0, '
    '"llamada_herramienta": null o {"nombre": '
    '"crear_reclamo|crear_reserva|consultar_estado_caso", "parametros": {...}}, '
    '"respuesta": "texto para el ciudadano"}\n\n'
    'Reglas:\n'
    '- Tratá al ciudadano de "usted", con lenguaje simple y claro, sin tecnicismos.\n'
    '- "llamada_herramienta" va en null si falta un dato necesario para ejecutar la '
    'acción; en ese caso "respuesta" repregunta justo lo que falta.\n'
    '- "tramite" nunca lleva llamada_herramienta: todavía no existe esa tool, '
    'siempre escala a un operador.\n'
    '- Nunca inventes un parámetro que el ciudadano no haya dicho.\n'
    '- "respuesta" en texto plano: sin markdown, sin asteriscos ni guiones de lista.'
)


def cargar_jsonl(ruta):
    return [json.loads(linea) for linea in open(ruta, encoding="utf-8")]


def a_conversacion(ejemplo):
    salida = {
        "intencion": ejemplo["intencion"],
        "confianza": ejemplo["confianza"],
        "llamada_herramienta": ejemplo["llamada_herramienta"],
        "respuesta": ejemplo["respuesta"],
    }
    return {
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": ejemplo["instruccion"]},
            {"role": "assistant", "content": json.dumps(salida, ensure_ascii=False)},
        ]
    }


In [ ]:
from datasets import Dataset

train_rows = [a_conversacion(e) for e in cargar_jsonl("data/train.jsonl")]
val_rows = [a_conversacion(e) for e in cargar_jsonl("data/validacion.jsonl")]

dataset_train = Dataset.from_list(train_rows)
dataset_val = Dataset.from_list(val_rows)


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    textos = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": textos}


dataset_train = dataset_train.map(formatting_prompts_func, batched=True)
dataset_val = dataset_val.map(formatting_prompts_func, batched=True)

print(f"train: {len(dataset_train)}, validacion: {len(dataset_val)}")
print("--- ejemplo ---")
print(dataset_train[0]["text"])


## 6. Entrenamiento

LR 2e-4, scheduler coseno, 3 épocas, batch efectivo 16 (4 por dispositivo × 4 de
acumulación de gradiente) — igual que el resto de los hiperparámetros, tal como
quedaron comprometidos en la tesis.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

BATCH_MICRO = 4
GRAD_ACUM = 4  # 4 x 4 = 16

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    eval_dataset=dataset_val,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    packing=False,
    args=SFTConfig(
        per_device_train_batch_size=BATCH_MICRO,
        gradient_accumulation_steps=GRAD_ACUM,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        weight_decay=0.001,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

# Enmascara la pérdida sobre el prompt (system+user): sólo se entrena a predecir
# el turno del asistente, no a memorizar la instrucción del ciudadano.
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer_stats = trainer.train()


## 7. Guardar y exportar a GGUF (para Ollama)

In [ ]:
model.save_pretrained("qwen_omnicivitas_lora")
tokenizer.save_pretrained("qwen_omnicivitas_lora")

# q4_k_m: cuantización GGUF de 4 bits estándar para servir con Ollama. No es
# exactamente la misma cuantización NF4 que se usó para entrenar (esa es sólo
# para entrenamiento); es la conversión equivalente para inferencia servida.
model.save_pretrained_gguf(
    "qwen_omnicivitas_gguf", tokenizer, quantization_method="q4_k_m"
)

print("Descargar la carpeta qwen_omnicivitas_gguf/ — ahí está el .gguf para Ollama.")


## 8. Evaluación sobre el held-out de tool calling

Mide los tres indicadores que compromete la tesis (`chapter04.tex`, "Entrenamiento
del modelo de lenguaje"), sobre `prueba_tool_calling.jsonl` — el conjunto que nunca
participó del entrenamiento ni de la validación:

- Exactitud de clasificación de intención (objetivo ≥ 90%).
- Falsos positivos: confianza ≥ 0,6 con categoría incorrecta (el caso que
  RF-06 deja pasar sin derivar — más grave que un falso negativo).
- Confiabilidad de tool calling: herramienta Y parámetros exactos
  (objetivo ≥ 95%).

In [ ]:
import re

FastLanguageModel.for_inference(model)


def generar(mensaje):
    convo = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": mensaje},
    ]
    texto = tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(texto, return_tensors="pt").to("cuda")
    salida = model.generate(
        **entradas, max_new_tokens=300, temperature=0.1, do_sample=False
    )
    return tokenizer.decode(
        salida[0][entradas["input_ids"].shape[1]:], skip_special_tokens=True
    )


def parsear(texto):
    try:
        return json.loads(texto.strip())
    except json.JSONDecodeError:
        coincidencia = re.search(r"\{.*\}", texto, re.DOTALL)
        if coincidencia:
            try:
                return json.loads(coincidencia.group(0))
            except json.JSONDecodeError:
                return None
        return None


In [ ]:
prueba = cargar_jsonl("data/prueba_tool_calling.jsonl")

correctos_intencion = 0
falsos_positivos = 0
correctos_tool = 0
sin_json_valido = 0
detalle = []

for ejemplo in prueba:
    crudo = generar(ejemplo["instruccion"])
    pred = parsear(crudo)
    if pred is None:
        sin_json_valido += 1
        detalle.append({"instruccion": ejemplo["instruccion"], "error": "sin JSON válido", "crudo": crudo})
        continue

    intencion_ok = pred.get("intencion") == ejemplo["intencion"]
    correctos_intencion += int(intencion_ok)
    if not intencion_ok and pred.get("confianza", 0) >= 0.6:
        falsos_positivos += 1

    tool_ok = pred.get("llamada_herramienta") == ejemplo["llamada_herramienta"]
    correctos_tool += int(tool_ok)

    detalle.append(
        {
            "instruccion": ejemplo["instruccion"],
            "intencion_esperada": ejemplo["intencion"],
            "intencion_predicha": pred.get("intencion"),
            "intencion_ok": intencion_ok,
            "tool_ok": tool_ok,
        }
    )

total = len(prueba)
print(f"Total evaluado: {total}")
print(f"Sin JSON válido: {sin_json_valido}")
print(f"Exactitud de intención: {correctos_intencion / total:.1%} ({correctos_intencion}/{total})")
print(f"Falsos positivos (confianza >= 0.6, categoría incorrecta): {falsos_positivos}")
print(f"Confiabilidad de tool calling (herramienta Y parámetros exactos): {correctos_tool / total:.1%} ({correctos_tool}/{total})")


## Próximo paso (fuera de este notebook)

Con el `.gguf` descargado: armar un `Modelfile` de Ollama, `ollama create
omnicivitas-qwen -f Modelfile`, y cambiar `app/agents/llm.py` para apuntar a ese
endpoint local compatible con OpenAI en vez de la API de Claude — es la tarea de
integración que sigue pendiente en `finetuning/DEVLOG.md`.